# 2. Data Cleaning & Pre-processing

## Data Cleaning
Explain how missing values were handled, duplicates removed, and inconsistencies corrected.

Example:
Missing values were checked and handled using ... . Duplicate records were removed to avoid bias in the analysis. Inconsistent values were standardized to ensure data quality.

## Feature Engineering
Describe any new variables created.

Example:
New features such as hour, day_of_week, month, and is_weekend were created because time-related patterns strongly influence traffic volume.

## Data Transformation
Explain scaling, encoding, or normalization steps.

Example:
Categorical variables were encoded using one-hot encoding, and numerical variables were scaled where necessary to ensure stable model performance.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/traffic_volume.csv')

In [2]:
df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   holiday              61 non-null     object 
 1   temp                 48204 non-null  float64
 2   rain_1h              48204 non-null  float64
 3   snow_1h              48204 non-null  float64
 4   clouds_all           48204 non-null  int64  
 5   weather_main         48204 non-null  object 
 6   weather_description  48204 non-null  object 
 7   date_time            48204 non-null  object 
 8   traffic_volume       48204 non-null  int64  
dtypes: float64(3), int64(2), object(4)
memory usage: 3.3+ MB


In [4]:
df.isnull().sum()

holiday                48143
temp                       0
rain_1h                    0
snow_1h                    0
clouds_all                 0
weather_main               0
weather_description        0
date_time                  0
traffic_volume             0
dtype: int64

## holiday and date_time cleaning

In [5]:
df['holiday'].unique()

array([nan, 'Columbus Day', 'Veterans Day', 'Thanksgiving Day',
       'Christmas Day', 'New Years Day', 'Washingtons Birthday',
       'Memorial Day', 'Independence Day', 'State Fair', 'Labor Day',
       'Martin Luther King Jr Day'], dtype=object)

#### first we want to fillna the holiday column

In [6]:
df[df['holiday'].notna()][['date_time', 'holiday']]

,date_time,holiday
126,2012-10-08 00:00:00,Columbus Day
1123,2012-11-12 00:00:00,Veterans Day
1370,2012-11-22 00:00:00,Thanksgiving Day
2360,2012-12-25 00:00:00,Christmas Day
2559,2013-01-01 00:00:00,New Years Day
...,...,...
44441,2018-05-28 00:00:00,Memorial Day
45547,2018-07-04 00:00:00,Independence Day
46936,2018-08-23 00:00:00,State Fair
47330,2018-09-03 00:00:00,Labor Day


In [7]:
df['date_time_temp'] = pd.to_datetime(df['date_time'])
sample_day = df[df['holiday'] == 'Christmas Day']['date_time_temp'].dt.date.iloc[0]
df[df['date_time_temp'].dt.date == sample_day][['date_time', 'holiday']]

,date_time,holiday
2360,2012-12-25 00:00:00,Christmas Day
2361,2012-12-25 01:00:00,NaN
2362,2012-12-25 02:00:00,NaN
2363,2012-12-25 03:00:00,NaN
2364,2012-12-25 04:00:00,NaN
2365,2012-12-25 05:00:00,NaN
2366,2012-12-25 06:00:00,NaN
2367,2012-12-25 07:00:00,NaN
2368,2012-12-25 07:00:00,NaN
2369,2012-12-25 08:00:00,NaN


#### we noticed that holiday record only the 1st hour of the day is holiday not the fullday

In [8]:
df['date_time'] = pd.to_datetime(df['date_time'])

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   holiday              61 non-null     object        
 1   temp                 48204 non-null  float64       
 2   rain_1h              48204 non-null  float64       
 3   snow_1h              48204 non-null  float64       
 4   clouds_all           48204 non-null  int64         
 5   weather_main         48204 non-null  object        
 6   weather_description  48204 non-null  object        
 7   date_time            48204 non-null  datetime64[ns]
 8   traffic_volume       48204 non-null  int64         
 9   date_time_temp       48204 non-null  datetime64[ns]
dtypes: datetime64[ns](2), float64(3), int64(2), object(3)
memory usage: 3.7+ MB


In [10]:
df = df.drop(columns=['date_time_temp'])

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   holiday              61 non-null     object        
 1   temp                 48204 non-null  float64       
 2   rain_1h              48204 non-null  float64       
 3   snow_1h              48204 non-null  float64       
 4   clouds_all           48204 non-null  int64         
 5   weather_main         48204 non-null  object        
 6   weather_description  48204 non-null  object        
 7   date_time            48204 non-null  datetime64[ns]
 8   traffic_volume       48204 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(2), object(3)
memory usage: 3.3+ MB


In [12]:
df['hour'] = df['date_time'].dt.hour

In [13]:
df['day_of_week'] = df['date_time'].dt.day_name()

In [14]:
df['month'] = df['date_time'].dt.month

In [15]:
df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,hour,day_of_week,month
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545,9,Tuesday,10
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516,10,Tuesday,10
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767,11,Tuesday,10
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026,12,Tuesday,10
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918,13,Tuesday,10


In [16]:
df['date'] = df['date_time'].dt.date

In [17]:
df['date_time'].duplicated().sum()

7629

In [18]:
df[df['date_time'].duplicated(keep=False)].sort_values('date_time').head(10)

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,hour,day_of_week,month,date
178,NaN,281.25,0.0,0.0,99,Rain,light rain,2012-10-10 07:00:00,6793,7,Wednesday,10,2012-10-10
179,NaN,281.25,0.0,0.0,99,Drizzle,light intensity drizzle,2012-10-10 07:00:00,6793,7,Wednesday,10,2012-10-10
180,NaN,280.10,0.0,0.0,99,Rain,light rain,2012-10-10 08:00:00,6283,8,Wednesday,10,2012-10-10
181,NaN,280.10,0.0,0.0,99,Drizzle,light intensity drizzle,2012-10-10 08:00:00,6283,8,Wednesday,10,2012-10-10
182,NaN,279.61,0.0,0.0,99,Rain,light rain,2012-10-10 09:00:00,5680,9,Wednesday,10,2012-10-10
183,NaN,279.61,0.0,0.0,99,Drizzle,light intensity drizzle,2012-10-10 09:00:00,5680,9,Wednesday,10,2012-10-10
269,NaN,282.43,0.0,0.0,57,Drizzle,light intensity drizzle,2012-10-14 09:00:00,2685,9,Sunday,10,2012-10-14
270,NaN,282.43,0.0,0.0,57,Mist,mist,2012-10-14 09:00:00,2685,9,Sunday,10,2012-10-14
271,NaN,282.43,0.0,0.0,57,Haze,haze,2012-10-14 09:00:00,2685,9,Sunday,10,2012-10-14
272,NaN,282.33,0.0,0.0,57,Drizzle,light intensity drizzle,2012-10-14 10:00:00,3370,10,Sunday,10,2012-10-14


#### droping duplicates

In [19]:
df.shape

(48204, 13)

In [20]:
df = df.drop_duplicates()

In [21]:
df.shape

(48187, 13)

In [22]:
df['weather_main'].value_counts()

weather_main
Clouds          15158
Clear           13384
Mist             5949
Rain             5672
Snow             2875
Drizzle          1820
Haze             1360
Thunderstorm     1033
Fog               912
Smoke              20
Squall              4
Name: count, dtype: int64

#### if same day, same hour has more than one weather?, we choose the worst

In [23]:
weather_priority = {
    'Squall': 0,
    'Thunderstorm': 1,
    'Snow': 2,
    'Fog': 3,
    'Rain': 4,
    'Drizzle': 5,
    'Haze': 6,
    'Smoke': 7,
    'Mist': 8,
    'Clouds': 9,
    'Clear': 10
}

In [24]:
df['weather_priority'] = df['weather_main'].map(weather_priority) # temp column

In [25]:
df = df.sort_values(['date_time', 'weather_priority'])
df = df.drop_duplicates(subset='date_time', keep='first')

In [26]:
df = df.drop(columns=['weather_priority'])

In [27]:
df.shape

(40575, 13)

In [28]:
df['date_time'].duplicated().sum()

0

In [29]:
df.duplicated().sum()

0

#### Nice

In [30]:
holiday_map = df[df['holiday'].notna()].set_index('date')['holiday']
holiday_map.index.duplicated().sum()

0

In [31]:
df['holiday'] = df['date'].map(holiday_map)

In [32]:
df['holiday'] = df['holiday'].fillna('Not Holiday')

In [33]:
df['holiday'].value_counts()

holiday
Not Holiday                  39372
Independence Day               120
State Fair                     119
Labor Day                      118
Memorial Day                   117
Washingtons Birthday           115
Christmas Day                  113
New Years Day                  112
Veterans Day                   108
Thanksgiving Day               107
Columbus Day                   105
Martin Luther King Jr Day       69
Name: count, dtype: int64

In [34]:
df = df.drop(columns=['date'])

In [35]:
df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,hour,day_of_week,month
0,Not Holiday,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545,9,Tuesday,10
1,Not Holiday,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516,10,Tuesday,10
2,Not Holiday,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767,11,Tuesday,10
3,Not Holiday,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026,12,Tuesday,10
4,Not Holiday,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918,13,Tuesday,10


In [36]:
df["hour"].unique()

array([ 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23,  0,  1,
        2,  3,  4,  5,  6,  8,  7])

## temp

In [37]:
df['temp'].describe()

count    40575.000000
mean       281.316750
std         13.816604
min          0.000000
25%        271.840000
50%        282.860000
75%        292.280000
max        310.070000
Name: temp, dtype: float64

In [38]:
(df['temp'] == 0).sum()

10

In [39]:
df[df['temp'] == 0]

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,hour,day_of_week,month
11898,Not Holiday,0.0,0.0,0.0,0,Clear,sky is clear,2014-01-31 03:00:00,361,3,Friday,1
11899,Not Holiday,0.0,0.0,0.0,0,Clear,sky is clear,2014-01-31 04:00:00,734,4,Friday,1
11900,Not Holiday,0.0,0.0,0.0,0,Clear,sky is clear,2014-01-31 05:00:00,2557,5,Friday,1
11901,Not Holiday,0.0,0.0,0.0,0,Clear,sky is clear,2014-01-31 06:00:00,5150,6,Friday,1
11946,Not Holiday,0.0,0.0,0.0,0,Clear,sky is clear,2014-02-02 03:00:00,291,3,Sunday,2
11947,Not Holiday,0.0,0.0,0.0,0,Clear,sky is clear,2014-02-02 04:00:00,284,4,Sunday,2
11948,Not Holiday,0.0,0.0,0.0,0,Clear,sky is clear,2014-02-02 05:00:00,434,5,Sunday,2
11949,Not Holiday,0.0,0.0,0.0,0,Clear,sky is clear,2014-02-02 06:00:00,739,6,Sunday,2
11950,Not Holiday,0.0,0.0,0.0,0,Clear,sky is clear,2014-02-02 07:00:00,962,7,Sunday,2
11951,Not Holiday,0.0,0.0,0.0,0,Clear,sky is clear,2014-02-02 08:00:00,1670,8,Sunday,2


In [40]:
df = df[df['temp'] != 0]

In [41]:
df.shape

(40565, 12)

## rain , snow and clouds

In [42]:
df['rain_1h'].describe()

count    40565.000000
mean         0.318710
std         48.818657
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max       9831.300000
Name: rain_1h, dtype: float64

##### max: 9831.300000 **SOOO WEIRD** 

In [43]:
df['snow_1h'].describe()

count    40565.000000
mean         0.000117
std          0.005677
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          0.510000
Name: snow_1h, dtype: float64

In [44]:
df[df["rain_1h"]>1000]

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,hour,day_of_week,month
24872,Not Holiday,302.11,9831.3,0.0,75,Rain,very heavy rain,2016-07-11 17:00:00,5535,17,Monday,7


In [45]:
df = df[df['rain_1h'] <= 1000]

In [46]:
df['rain_1h'].describe()

count    40564.000000
mean         0.076353
std          0.769729
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         55.630000
Name: rain_1h, dtype: float64

In [47]:
df['clouds_all'].describe()

count    40564.000000
mean        44.212306
std         38.681034
min          0.000000
25%          1.000000
50%         40.000000
75%         90.000000
max        100.000000
Name: clouds_all, dtype: float64

In [48]:
df['weather_main'].nunique()

11

In [49]:
df['weather_description'].nunique()

35

In [50]:
df.groupby('weather_main')['weather_description'].unique()

weather_main
Clear                                [sky is clear, Sky is Clear]
Clouds          [scattered clouds, broken clouds, overcast clo...
Drizzle         [light intensity drizzle, drizzle, heavy inten...
Fog                                                         [fog]
Haze                                                       [haze]
Mist                                                       [mist]
Rain            [light rain, proximity shower rain, moderate r...
Smoke                                                     [smoke]
Snow            [heavy snow, snow, light rain and snow, light ...
Squall                                                  [SQUALLS]
Thunderstorm    [proximity thunderstorm, thunderstorm with lig...
Name: weather_description, dtype: object

#### Dropping `weather_description` — keeping `weather_main` only since it's just a more detailed, redundant version of it.

In [51]:
df = df.drop(columns=['weather_description'])

In [52]:
df.shape

(40564, 11)

In [53]:
df['weather_main'].unique()

array(['Clouds', 'Clear', 'Rain', 'Drizzle', 'Mist', 'Fog',
       'Thunderstorm', 'Haze', 'Snow', 'Squall', 'Smoke'], dtype=object)

In [54]:
df = df.reset_index(drop=True)

In [55]:
df.isna().sum()

holiday           0
temp              0
rain_1h           0
snow_1h           0
clouds_all        0
weather_main      0
date_time         0
traffic_volume    0
hour              0
day_of_week       0
month             0
dtype: int64

In [56]:
df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,date_time,traffic_volume,hour,day_of_week,month
0,Not Holiday,288.28,0.0,0.0,40,Clouds,2012-10-02 09:00:00,5545,9,Tuesday,10
1,Not Holiday,289.36,0.0,0.0,75,Clouds,2012-10-02 10:00:00,4516,10,Tuesday,10
2,Not Holiday,289.58,0.0,0.0,90,Clouds,2012-10-02 11:00:00,4767,11,Tuesday,10
3,Not Holiday,290.13,0.0,0.0,90,Clouds,2012-10-02 12:00:00,5026,12,Tuesday,10
4,Not Holiday,291.14,0.0,0.0,75,Clouds,2012-10-02 13:00:00,4918,13,Tuesday,10


## Cyclical Encoding

In [57]:
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

In [58]:
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

In [59]:
df[['hour', 'hour_sin', 'hour_cos', 'month', 'month_sin', 'month_cos']].head()

,hour,hour_sin,hour_cos,month,month_sin,month_cos
0,9,7.071068e-01,-0.707107,10,-0.866025,0.5
1,10,5.000000e-01,-0.866025,10,-0.866025,0.5
2,11,2.588190e-01,-0.965926,10,-0.866025,0.5
3,12,1.224647e-16,-1.000000,10,-0.866025,0.5
4,13,-2.588190e-01,-0.965926,10,-0.866025,0.5


## is_rush_hour?

In [60]:
df.groupby('hour')['traffic_volume'].mean().sort_values(ascending=False)

hour
16    5708.614528
17    5350.719784
15    5271.737823
14    4957.022982
7     4772.110114
13    4741.578004
12    4739.118794
8     4595.331754
11    4498.458908
9     4394.778523
18    4306.467301
10    4204.876833
6     4173.518739
19    3311.021390
20    2869.703920
21    2699.360915
22    2224.418848
5     2104.389780
23    1490.710345
0      844.854324
4      701.543798
1      521.318713
2      392.637006
3      373.266947
Name: traffic_volume, dtype: float64

#### Skipping `is_rush_hour`
Traffic volume changes gradually across daytime hours with no clear jump, so a binary flag would lose detail that `hour_sin`/`hour_cos` already capture.

In [61]:
df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,date_time,traffic_volume,hour,day_of_week,month,hour_sin,hour_cos,month_sin,month_cos
0,Not Holiday,288.28,0.0,0.0,40,Clouds,2012-10-02 09:00:00,5545,9,Tuesday,10,7.071068e-01,-0.707107,-0.866025,0.5
1,Not Holiday,289.36,0.0,0.0,75,Clouds,2012-10-02 10:00:00,4516,10,Tuesday,10,5.000000e-01,-0.866025,-0.866025,0.5
2,Not Holiday,289.58,0.0,0.0,90,Clouds,2012-10-02 11:00:00,4767,11,Tuesday,10,2.588190e-01,-0.965926,-0.866025,0.5
3,Not Holiday,290.13,0.0,0.0,90,Clouds,2012-10-02 12:00:00,5026,12,Tuesday,10,1.224647e-16,-1.000000,-0.866025,0.5
4,Not Holiday,291.14,0.0,0.0,75,Clouds,2012-10-02 13:00:00,4918,13,Tuesday,10,-2.588190e-01,-0.965926,-0.866025,0.5


##### all good

In [62]:
df.to_csv('../data/processed/traffic_volume_cleaned.csv', index=False)

In [64]:
clean_data = pd.read_csv('../data/processed/traffic_volume_cleaned.csv').head()

In [65]:
clean_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   holiday         5 non-null      object 
 1   temp            5 non-null      float64
 2   rain_1h         5 non-null      float64
 3   snow_1h         5 non-null      float64
 4   clouds_all      5 non-null      int64  
 5   weather_main    5 non-null      object 
 6   date_time       5 non-null      object 
 7   traffic_volume  5 non-null      int64  
 8   hour            5 non-null      int64  
 9   day_of_week     5 non-null      object 
 10  month           5 non-null      int64  
 11  hour_sin        5 non-null      float64
 12  hour_cos        5 non-null      float64
 13  month_sin       5 non-null      float64
 14  month_cos       5 non-null      float64
dtypes: float64(7), int64(4), object(4)
memory usage: 728.0+ bytes


In [67]:
clean_data.isnull().sum()

holiday           0
temp              0
rain_1h           0
snow_1h           0
clouds_all        0
weather_main      0
date_time         0
traffic_volume    0
hour              0
day_of_week       0
month             0
hour_sin          0
hour_cos          0
month_sin         0
month_cos         0
dtype: int64